### Preprocessing

In [ ]:
stats_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(root='plant_img_train', transform=stats_transform)
dataloader = DataLoader(train_dataset, batch_size=32, shuffle=False, num_workers=4)

mean = torch.zeros(3)
std = torch.zeros(3)
n_pixels = 0

with torch.no_grad():
    for images, _ in dataloader:
        batch_pixels = images.size(0) * images.size(2) * images.size(3)
        mean += images.sum(dim=[0, 2, 3])
        std += (images ** 2).sum(dim=[0, 2, 3])
        n_pixels += batch_pixels

mean /= n_pixels
std = (std / n_pixels - mean ** 2).sqrt()

print('Mean:', mean)
print('Std:', std)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean.tolist(), std=std.tolist()),
    transforms.RandomErasing(p=0.25)
])

eval_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean.tolist(), std=std.tolist()),
])

train_dataset = datasets.ImageFolder(root='plant_img_train', transform=train_transform)
val_dataset = datasets.ImageFolder(root='plant_img_val', transform=eval_transform)
test_dataset = datasets.ImageFolder(root='plant_img_test', transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)